# RCC Subtype Classification - Statistical Significance Testing
### BME 515 Final Project
McNemar's test for pairwise model comparison.
Tests whether differences in per-slide classification accuracy between models are statistically significant.

Required files (place all in BASE_DIR):
- meanpool_5class_output/meanpool_test_predictions.csv  (from Step 3)
- abmil_test_predictions.csv
- clam_sb_test_predictions.csv
- clam_mb_test_predictions.csv

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
from itertools import combinations
from statsmodels.stats.contingency_tables import mcnemar

BASE_DIR   = Path('/Users/zhangruikun/Desktop/BME515')
OUTPUT_DIR = BASE_DIR / 'meanpool_5class_output'
OUTPUT_DIR.mkdir(exist_ok=True)

print('Libraries loaded')

In [ ]:
# load all prediction files and check columns
files = {
    'Mean Pool MLP': BASE_DIR / 'meanpool_5class_output' / 'meanpool_test_predictions.csv',
    'ABMIL':         BASE_DIR / 'abmil_test_predictions.csv',
    'CLAM-SB':       BASE_DIR / 'clam_sb_test_predictions.csv',
    'CLAM-MB':       BASE_DIR / 'clam_mb_test_predictions.csv',
}

raw = {}
for name, path in files.items():
    if not path.exists():
        print(f'MISSING: {path}')
        continue
    df = pd.read_csv(path)
    print(f'{name}: {len(df)} slides | columns: {list(df.columns)}')
    raw[name] = df

In [ ]:
# NOTE: check column names from output above and adjust if needed
# For Mean Pool: y_true, y_pred_mlp
# For CLAM/ABMIL: adjust column name below based on what you see above

def get_correct(df, pred_col, true_col='y_true'):
    """Returns boolean array: True if prediction is correct."""
    return (df[pred_col] == df[true_col]).values

# build correct/incorrect array per model
# adjust pred_col names based on actual column names in each CSV
correct = {}

# Mean Pool MLP
correct['Mean Pool MLP'] = get_correct(raw['Mean Pool MLP'], pred_col='y_pred_mlp')

# ABMIL, CLAM-SB, CLAM-MB — check column names from cell above
# common column names in CLAM output: 'pred', 'y_pred', 'predicted_label'
# update these after checking the output above
for model in ['ABMIL', 'CLAM-SB', 'CLAM-MB']:
    df = raw[model]
    # try common column names
    for col in ['pred', 'y_pred', 'predicted_label', 'prediction']:
        if col in df.columns:
            correct[model] = get_correct(df, pred_col=col)
            print(f'{model}: using column "{col}"')
            break
    else:
        print(f'{model}: could not find pred column — check columns above and update manually')

print('\nCorrect predictions per model:')
for name, arr in correct.items():
    print(f'  {name}: {arr.sum()}/{len(arr)} ({arr.mean()*100:.1f}%)')

In [ ]:
# McNemar's test — all pairwise comparisons
# null hypothesis: two models make errors on the same slides
# p < 0.05 means significantly different error patterns

model_names = list(correct.keys())
results = []

for m1, m2 in combinations(model_names, 2):
    c1 = correct[m1]
    c2 = correct[m2]

    # contingency table
    # b = m1 correct, m2 wrong
    # c = m1 wrong, m2 correct
    b = ((c1 == True)  & (c2 == False)).sum()
    c = ((c1 == False) & (c2 == True)).sum()

    table = [[0, b], [c, 0]]  # McNemar only uses off-diagonal

    # exact=True recommended for small samples (n < 25 discordant pairs)
    n_discordant = b + c
    use_exact = n_discordant < 25
    result = mcnemar([[c1.sum(), b], [c, (~c1).sum()]], exact=use_exact)

    results.append({
        'Model A': m1,
        'Model B': m2,
        'A correct, B wrong': b,
        'B correct, A wrong': c,
        'Discordant pairs': n_discordant,
        'p-value': round(result.pvalue, 4),
        'Significant (p<0.05)': 'Yes' if result.pvalue < 0.05 else 'No',
        'Test type': 'Exact' if use_exact else 'Chi-square'
    })

results_df = pd.DataFrame(results)
print(results_df.to_string(index=False))

In [ ]:
# save results
results_df.to_csv(str(OUTPUT_DIR / 'mcnemar_results.csv'), index=False)
print(f'Saved mcnemar_results.csv')

print('\n--- Summary ---')
sig = results_df[results_df['Significant (p<0.05)'] == 'Yes']
not_sig = results_df[results_df['Significant (p<0.05)'] == 'No']
print(f'Significant pairs: {len(sig)}')
print(f'Non-significant pairs: {len(not_sig)}')
if len(sig) > 0:
    print('\nSignificant differences:')
    for _, row in sig.iterrows():
        print(f'  {row["Model A"]} vs {row["Model B"]}: p={row["p-value"]}')